<a href="https://colab.research.google.com/github/kader-xai/llm-from-scratch/blob/main/notebooks/answer/Chapter12_trainer_answer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧱 **LLM From Scratch · Chapter 12 — Trainer — the training loop**

> ✅ Answer notebook — full working code.

## Goal

Train the GPT on Tiny Shakespeare. The loop is the same four steps as any PyTorch model:
`forward → loss → backward → step`, with **AdamW** as the optimizer. We also estimate validation
loss periodically. On CPU this is slow but works; on a Colab T4 GPU it's a couple of minutes.

In [ ]:
# ── Setup (run first) ─────────────────────────────────────────────────
# Pure PyTorch — no extra installs needed beyond torch (preinstalled on Colab).
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

In [ ]:
# Download the training corpus (Tiny Shakespeare, ~1 MB) if not present.
import os, urllib.request
if not os.path.exists("input.txt"):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, "input.txt")
text = open("input.txt").read()
print(f"corpus: {len(text):,} characters")
print(text[:200])

In [ ]:
# Tokenizer + data (Chapter 01) and the GPT (Chapter 11), compact + self-contained
chars=sorted(set(text)); vocab_size=len(chars)
stoi={c:i for i,c in enumerate(chars)}; itos={i:c for i,c in enumerate(chars)}
encode=lambda s:[stoi[c] for c in s]; decode=lambda l:"".join(itos[i] for i in l)
data=torch.tensor(encode(text),dtype=torch.long); n=int(0.9*len(data))
train_data,val_data=data[:n],data[n:]
block_size,batch_size=32,32
def get_batch(split):
    d=train_data if split=="train" else val_data
    ix=torch.randint(len(d)-block_size,(batch_size,))
    x=torch.stack([d[i:i+block_size] for i in ix]); y=torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device),y.to(device)

class Head(nn.Module):
    def __init__(s,C,hs,bs):
        super().__init__(); s.key=nn.Linear(C,hs,bias=False);s.query=nn.Linear(C,hs,bias=False);s.value=nn.Linear(C,hs,bias=False)
        s.register_buffer("tril",torch.tril(torch.ones(bs,bs)));s.scale=hs**-0.5
    def forward(s,x):
        B,T,C=x.shape;k,q,v=s.key(x),s.query(x),s.value(x)
        a=(q@k.transpose(-2,-1)*s.scale).masked_fill(s.tril[:T,:T]==0,float("-inf"));return F.softmax(a,-1)@v
class MHA(nn.Module):
    def __init__(s,C,nh,bs):super().__init__();s.heads=nn.ModuleList([Head(C,C//nh,bs) for _ in range(nh)]);s.proj=nn.Linear(C,C)
    def forward(s,x):return s.proj(torch.cat([h(x) for h in s.heads],-1))
class FF(nn.Module):
    def __init__(s,C):super().__init__();s.net=nn.Sequential(nn.Linear(C,4*C),nn.GELU(),nn.Linear(4*C,C))
    def forward(s,x):return s.net(x)
class Block(nn.Module):
    def __init__(s,C,nh,bs):super().__init__();s.sa=MHA(C,nh,bs);s.ff=FF(C);s.ln1=nn.LayerNorm(C);s.ln2=nn.LayerNorm(C)
    def forward(s,x):x=x+s.sa(s.ln1(x));x=x+s.ff(s.ln2(x));return x
class GPT(nn.Module):
    def __init__(s,vocab_size,C=128,nh=4,nl=4,bs=32):
        super().__init__();s.block_size=bs;s.tok=nn.Embedding(vocab_size,C);s.pos=nn.Embedding(bs,C)
        s.blocks=nn.Sequential(*[Block(C,nh,bs) for _ in range(nl)]);s.ln_f=nn.LayerNorm(C);s.head=nn.Linear(C,vocab_size)
    def forward(s,ids,targets=None):
        B,T=ids.shape;x=s.tok(ids)+s.pos(torch.arange(T,device=ids.device));x=s.blocks(x)
        logits=s.head(s.ln_f(x));loss=None
        if targets is not None: loss=F.cross_entropy(logits.view(-1,logits.size(-1)),targets.view(-1))
        return logits,loss
    @torch.no_grad()
    def generate(s,ids,n):
        for _ in range(n):
            l,_=s(ids[:,-s.block_size:]);p=F.softmax(l[:,-1,:],-1);ids=torch.cat([ids,torch.multinomial(p,1)],1)
        return ids
model=GPT(vocab_size).to(device)
print("params:",sum(p.numel() for p in model.parameters()))

## The training loop

`max_iters=2000` gives readable output in a few minutes on a T4. Lower it to `500` for a quick test
on CPU. Watch the loss drop from ~4.2 (random) toward ~1.6.

In [ ]:
@torch.no_grad()
def estimate_loss(iters=50):
    model.eval(); out={}
    for split in ("train","val"):
        losses=torch.zeros(iters)
        for k in range(iters):
            xb,yb=get_batch(split); _,l=model(xb,yb); losses[k]=l.item()
        out[split]=losses.mean().item()
    model.train(); return out

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
max_iters = 2000
for it in range(max_iters):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)        # forward
    optimizer.zero_grad(set_to_none=True)
    loss.backward()                     # backward
    optimizer.step()                    # update
    if it % 500 == 0 or it == max_iters - 1:
        l = estimate_loss()
        print(f"iter {it:>4}: train {l['train']:.3f} | val {l['val']:.3f}")

✅ Loss is falling — the model is learning Shakespeare. Final chapter: generate text.